Compruebo que Raw existe

In [8]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


spark = get_spark("novashop-m02")
print(RAW.exists())

ROOT    /workspaces/python-pyspark-201
RAW     /workspaces/python-pyspark-201/data/raw existe: True
STAGING /workspaces/python-pyspark-201/data/staging
CURATED /workspaces/python-pyspark-201/data/curated
True


Leo los dos CSVs

In [9]:
customers = spark.read.option("header", True).csv(str(RAW / "customers.csv"))
orders = spark.read.option("header", True).csv(str(RAW / "orders.csv"))
print("customers", customers.count(), "orders", orders.count())
orders.printSchema()
orders.show(3, truncate=False)

customers 250 orders 800
root
 |-- OrderId: string (nullable = true)
 |-- CustomerId: string (nullable = true)
 |-- OrderDate: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Channel: string (nullable = true)

+-------+----------+-------------------+------+-------+
|OrderId|CustomerId|OrderDate          |Status|Channel|
+-------+----------+-------------------+------+-------+
|O00001 |NULL      |2024-04-16 20:00:00|paid  |app    |
|O00002 |NULL      |2024-12-04 02:00:00|paid  |store  |
|O00003 |NULL      |2024-03-29 17:00:00|paid  |WEB    |
+-------+----------+-------------------+------+-------+
only showing top 3 rows



Creamos array de Json

In [10]:
products = spark.read.option("multiLine", True).json(str(RAW / "products.json"))
events = spark.read.json(str(RAW / "events.jsonl"))
print("products", products.count(), "events", events.count())
products.printSchema()
events.printSchema()

products 60 events 2500
root
 |-- category: string (nullable = true)
 |-- listPrice: string (nullable = true)
 |-- name: string (nullable = true)
 |-- productId: string (nullable = true)

root
 |-- customer_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- page: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- ts: string (nullable = true)



In [11]:
events.show()

+-----------+--------+-----------+---------+----------+----------+-------------------+
|customer_id|event_id| event_type|     page|product_id|session_id|                 ts|
+-----------+--------+-----------+---------+----------+----------+-------------------+
|       NULL| E000001|  page_view| /catalog|      P038|     S0806|2024-08-11T23:46:00|
|       NULL| E000002|  page_view|/checkout|      P058|     S0636|2024-07-05T11:43:00|
|       NULL| E000003|  page_view|    /cart|      P009|     S0055|2024-07-24T22:03:00|
|       NULL| E000004|  page_view|/checkout|      P030|     S0474|2024-05-26T16:57:00|
|       NULL| E000005|  page_view| /product|      P054|     S0057|2024-11-01T13:01:00|
|       NULL| E000006|  page_view|/checkout|      P007|     S0342|2024-12-25T01:18:00|
|       NULL| E000007|add_to_cart| /product|      P056|     S0494|2024-10-26T06:13:00|
|       NULL| E000008|   purchase|/checkout|      P048|     S0599|2024-01-23T03:50:00|
|       NULL| E000009|   purchase|/checkout

In [12]:
items = spark.read.option("header", True).csv(str(RAW / "order_items.csv"))
print(items.count())  # 2046
items.show(3)


2046
+--------+----------+---+----------+--------+
|order_id|product_id|qty|unit_price|discount|
+--------+----------+---+----------+--------+
|  O00001|      P053|  3|    114.41|    0.00|
|  O00002|      P016|  1|     90.86|    0.20|
|  O00002|      P007|  1|     16.79|    0.05|
+--------+----------+---+----------+--------+
only showing top 3 rows

